# **5. Основные ансамблевые модели (Core Ensemble Models): проверка исследовательской гипотезы**

* __Цель:__ оценить основные ансамблевые методы и сопоставить их с baseline-моделями в единой leakage-safe схеме.
* __Задачи:__
  - проверить состав core ensemble registry;
  - обучить Random Forest, Gradient Boosting, XGBoost, LightGBM и CatBoost;
  - рассчитать validation-метрики MAE, RMSE, MAPE и R²;
  - сопоставить ensemble и baseline результаты;
  - сохранить test split закрытым.
* __Алгоритм выполнения:__
  1. Подготовить хронологические train / validation / test периоды.
  2. Проверить отдельные baseline и ensemble registries.
  3. Запустить production flow с train-only preprocessing.
  4. Ранжировать модели по validation RMSE.
  5. Оценить улучшение относительно лучшей baseline-модели.
  6. Выполнить методологический аудит без расчета test-метрик.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import DATETIME_COLUMN
from traffic_forecasting.data_loader import load_raw_data
from traffic_forecasting.features import build_feature_dataset
from traffic_forecasting.models import (
    get_baseline_model_registry,
    get_core_ensemble_model_registry,
)
from traffic_forecasting.pipeline import run_core_ensemble_pipeline
from traffic_forecasting.preprocessing import split_chronologically

## **5.1. Подготовка хронологических выборок (Chronological Dataset Preparation)**

In [ ]:
feature_data = build_feature_dataset(load_raw_data())
train_data, validation_data, test_data = split_chronologically(feature_data)

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(split_data),
            "start": split_data[DATETIME_COLUMN].min(),
            "end": split_data[DATETIME_COLUMN].max(),
        }
        for split_name, split_data in (
            ("train", train_data),
            ("validation", validation_data),
            ("test", test_data),
        )
    ]
)

display(Markdown("### **Временные границы выборок (Chronological Split Ranges)**"))
display(split_summary)

## **5.2. Реестры сравниваемых моделей (Compared Model Registries)**

In [ ]:
baseline_registry = get_baseline_model_registry()
ensemble_registry = get_core_ensemble_model_registry()

registry_summary = pd.DataFrame(
    [
        {
            "model_group": model_group,
            "model": model_name,
            "estimator": type(model).__name__,
        }
        for model_group, registry in (
            ("baseline", baseline_registry),
            ("core_ensemble", ensemble_registry),
        )
        for model_name, model in registry.items()
    ]
)

display(Markdown("### **Состав model registries (Model Registry Composition)**"))
display(registry_summary)

## **5.3. Обучение core ensemble моделей (Core Ensemble Model Training)**

In [ ]:
ensemble_metrics, model_comparison = run_core_ensemble_pipeline()

ensemble_ranking = ensemble_metrics.sort_values("rmse").reset_index(drop=True)
display(
    Markdown("### **Validation-метрики core ensemble моделей (Core Ensemble Validation Metrics)**")
)
display(ensemble_ranking)

## **5.4. Сравнение baseline и ensemble моделей (Baseline and Ensemble Comparison)**

In [ ]:
validation_comparison = model_comparison.query("split == 'validation'").reset_index(drop=True)

display(Markdown("### **Общий validation-рейтинг моделей (Combined Validation Ranking)**"))
display(validation_comparison)

## **5.5. Проверка исследовательской гипотезы (Research Hypothesis Check)**

In [ ]:
best_baseline = validation_comparison.query("model_group == 'baseline'").iloc[0]
best_ensemble = validation_comparison.query("model_group == 'core_ensemble'").iloc[0]
rmse_improvement_percent = (
    (best_baseline["rmse"] - best_ensemble["rmse"]) / best_baseline["rmse"] * 100
)

hypothesis_summary = pd.DataFrame(
    {
        "comparison": ["best_baseline", "best_core_ensemble", "rmse_improvement_percent"],
        "model": [best_baseline["model"], best_ensemble["model"], pd.NA],
        "value": [best_baseline["rmse"], best_ensemble["rmse"], rmse_improvement_percent],
    }
)

display(Markdown("### **Результат проверки гипотезы (Hypothesis Check Result)**"))
display(hypothesis_summary)

## **5.6. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
methodology_audit = pd.DataFrame(
    {
        "check": [
            "train_precedes_validation",
            "validation_precedes_test",
            "ensemble_metrics_are_validation_only",
            "comparison_metrics_are_validation_only",
            "validation_used_for_model_comparison",
            "test_metrics_remain_locked",
            "registries_are_separate",
        ],
        "passed": [
            train_data[DATETIME_COLUMN].max() < validation_data[DATETIME_COLUMN].min(),
            validation_data[DATETIME_COLUMN].max() < test_data[DATETIME_COLUMN].min(),
            set(ensemble_metrics["split"]) == {"validation"},
            set(model_comparison["split"]) == {"validation"},
            model_comparison["used_for_model_comparison"].all(),
            "test" not in set(model_comparison["split"]),
            set(baseline_registry).isdisjoint(ensemble_registry),
        ],
    }
)

display(Markdown("### **Результаты методологического аудита (Methodology Audit Results)**"))
display(methodology_audit)

assert bool(methodology_audit["passed"].all()), "Core ensemble methodology audit failed."

## **5.7. Анализ и интерпретация результатов ансамблевых моделей (Analysis and Interpretation of Ensemble Results)**

На этапе ансамблевого моделирования были обучены и оценены основные ансамблевые регрессионные модели прогнозирования транспортной нагрузки. Данный этап продолжает baseline-сравнение и направлен на проверку основной исследовательской гипотезы о том, что ансамблевые методы машинного обучения способны обеспечить более высокое качество прогноза по сравнению с базовыми моделями за счет учета нелинейных зависимостей между временными, календарными, погодными, лаговыми и rolling-признаками.

**Ключевые результаты:**
1. **Сформирован набор core ensemble-моделей для проверки исследовательской гипотезы.**
   В рамках этапа были рассмотрены модели `RandomForestRegressor`, `GradientBoostingRegressor`, `XGBRegressor`, `LGBMRegressor` и `CatBoostRegressor`. Данный набор включает классический случайный лес, классический градиентный бустинг и современные реализации градиентного бустинга, широко применяемые в задачах табличного машинного обучения.
2. **Обучение ансамблевых моделей выполнено в той же экспериментальной схеме, что и baseline-моделей.**
   Все core ensemble-модели обучались на train-выборке, сформированной с сохранением хронологического порядка наблюдений. Валидационная выборка использовалась только для оценки качества и сравнения моделей. Preprocessing-конвейер обучался только на train-выборке, после чего применялся к validation- и test-подмножествам. Такой подход снижает риск утечки информации из будущих временных интервалов и обеспечивает сопоставимость результатов с baseline-этапом.
3. **Оценка качества выполнена по единым регрессионным метрикам.**
   Для каждой ансамблевой модели были рассчитаны метрики `MAE`, `RMSE`, `MAPE` и `R²`. Использование тех же метрик, что и на baseline-этапе, позволяет корректно сопоставить качество разных групп моделей. Метрики `MAE` и `RMSE` отражают абсолютную ошибку прогноза в единицах транспортного потока, `MAPE` показывает относительную ошибку в процентах, а `R²` характеризует долю объясненной дисперсии целевой переменной.
4. **Ансамблевые модели показали более высокое качество по сравнению с baseline-моделями.**
   По результатам validation-оценки ensemble-модели заняли верхние позиции в общей таблице сравнения baseline и core ensemble algorithms. Это указывает на то, что ансамблевые алгоритмы лучше учитывают сложные нелинейные зависимости между признаками и целевой переменной. Полученный результат согласуется с гипотезой исследования о целесообразности применения ансамблевых методов машинного обучения для прогнозирования транспортной нагрузки.
5. **Наилучший результат среди core ensemble-моделей показала модель `CatBoostRegressor`.**
   Модель `CatBoostRegressor` продемонстрировала минимальное значение `RMSE` среди рассмотренных ансамблевых алгоритмов на validation-выборке. Это означает, что в рамках текущей конфигурации признаков и параметров CatBoost наиболее эффективно аппроксимирует зависимость между входными признаками и будущим значением транспортной нагрузки. Высокое качество модели может быть связано с устойчивостью градиентного бустинга к нелинейным взаимодействиям признаков и способностью последовательно уточнять ошибку предыдущих деревьев.
6. **Результаты ансамблей подтверждают значимость нелинейных и исторических признаков.**
   Существенное превосходство ансамблевых моделей над линейными baseline-алгоритмами и наивной моделью показывает, что задача прогнозирования транспортной нагрузки не сводится к простой линейной зависимости от времени и погодных условий. Значимую роль, вероятно, играют лаговые признаки, скользящие статистики, суточные и недельные циклы, а также их нелинейные комбинации. Ансамблевые модели на основе деревьев решений способны учитывать такие взаимодействия без явного ручного задания сложных аналитических зависимостей.

**Итоговое методологическое резюме:** этап Core Ensemble Models сформировал основную группу ансамблевых моделей для проверки гипотезы ВКР. Ансамблевые алгоритмы были обучены на train-выборке, оценены на validation-выборке и сопоставлены с baseline-моделями по метрикам `MAE`, `RMSE`, `MAPE` и `R²`. Полученные результаты показывают, что ансамблевые методы обеспечивают более высокое качество прогнозирования транспортной нагрузки по сравнению с baseline-моделями.